# Labwork 1 - PCA: Abalone dataset
**Full name:** ...  |  **Student ID:** ...

This notebook covers Part 1 and Part 2 for the **Abalone** dataset (UCI id 1). The structure is the same as in the Raisin notebook; the differences are the categorical feature `Sex` and the label `Rings` (number of rings, age ≈ Rings + 1.5).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
from scipy.stats import chi2_contingency, f_oneway

**Explanation:** we also import `chi2_contingency` (chi-square test for two categorical variables) and `f_oneway` (ANOVA: compares means across groups) to analyse the categorical feature in section 8.

## 1. Dataset description

In [ ]:
abalone = fetch_ucirepo(id=1)
X = abalone.data.features
y = abalone.data.targets
df = pd.concat([X, y], axis=1)

df.to_csv("../data/raw/abalone_raw.csv", index=False)
print(df.shape)
df.head()

**Explanation:** download the data, merge the table, save the raw copy.

**Description (to complete):** Abalone contains 4177 abalone samples with physical measurements (length, diameter, height, several weights) and sex. *(TODO: write the description in your own words and cite the UCI source.)*

## 2. Feature types

In [ ]:
print(df.dtypes)

**TODO - fill in the classification table:**

| Feature | Discrete / Continuous | Quantitative / Qualitative | Numerical / Categorical |
|---|---|---|---|
| Sex | ? | ? | ? |
| Length | ? | ? | ? |
| Diameter | ? | ? | ? |
| Height | ? | ? | ? |
| Whole_weight | ? | ? | ? |
| Shucked_weight | ? | ? | ? |
| Viscera_weight | ? | ? | ? |
| Shell_weight | ? | ? | ? |
| Rings | ? | ? | ? |

## 3. Data quality issues

In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Samples with Height = 0:", (df["Height"] == 0).sum())
print()
print(df["Sex"].value_counts())
df.describe()

**Explanation:** besides missing values and duplicates, we also check `Height = 0` (a height of zero is physically impossible, i.e. an *incorrect value*) and the labels of `Sex` (M, F, I = Infant) to see whether they are inconsistent.

In [ ]:
num_cols = df.select_dtypes(include="number").columns

outlier_count = {}
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_count[col] = ((df[col] < low) | (df[col] > high)).sum()

pd.Series(outlier_count, name="n_outliers")

In [ ]:
df[num_cols].plot(kind="box", subplots=True, layout=(3, 3), figsize=(14, 9))
plt.tight_layout()
plt.savefig("../figures/abalone_boxplot_before.png", dpi=150)
plt.show()

**Explanation:** count and plot the outliers with the IQR rule.

**TODO - discussion:** list the issues found (Height = 0, outliers in Height and the weight columns, ...) and explain how they affect the analysis.

## 4. Data cleaning

In [ ]:
df_clean = df.copy()

# 4.1 Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# 4.2 Remove rows with missing values (if any)
df_clean = df_clean.dropna()

# 4.3 Remove samples with Height = 0 (incorrect values)
df_clean = df_clean[df_clean["Height"] > 0]

# 4.4 Clip the outliers of the numerical columns, except Rings
feature_cols = [c for c in num_cols if c != "Rings"]
for col in feature_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    df_clean[col] = df_clean[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print("Before:", df.shape, "| After:", df_clean.shape)
df_clean.to_csv("../data/processed/abalone_clean.csv", index=False)

**Explanation:** samples with `Height = 0` are removed because they are incorrect data; outliers are only clipped to keep as many samples as possible. We do not clip `Rings` because it is the label.

**TODO:** compare before/after and justify your choices.

In [ ]:
comparison = pd.DataFrame({
    "mean_before": df[feature_cols].mean(),
    "mean_after": df_clean[feature_cols].mean(),
    "std_before": df[feature_cols].std(),
    "std_after": df_clean[feature_cols].std(),
})
comparison

## 5. Data integration
**TODO:** Abalone is a single table with no accompanying source. Write a few sentences explaining why integration is unnecessary (or describe it if you merged an extra source).

## 6. Data transformation

In [ ]:
# One-hot encode the categorical feature Sex
df_enc = pd.get_dummies(df_clean, columns=["Sex"], dtype=int)

# Standardize the numerical features (Rings excluded)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[feature_cols])

print(df_enc.head())

**Explanation:** `Sex` (M/F/I) is text, so it is *one-hot encoded* into 3 columns of 0/1 (there is no order, so we do not use the numbers 0, 1, 2). The numerical features are standardized.

**Note for PCA:** to keep things simple, the PCA below uses only the 7 numerical features; `Sex` is used to color the plots. **TODO:** justify this choice (PCA is based on variance and is not directly suited to one-hot variables).

## 7. Label and machine learning task
**TODO:** the label is `Rings` (number of rings on the shell, estimated age = Rings + 1.5). The task can be **regression** (predict Rings) or **classification** (age group). Add its practical meaning.

In [ ]:
df_clean["Rings"].describe()

## 8. Mean, variance, covariance, correlation

In [ ]:
features = df_clean[feature_cols]

stats = pd.DataFrame({"mean": features.mean(), "variance": features.var()})
print(stats)

print("\nCovariance matrix:")
print(features.cov().round(4))

corr = features.corr()

**Explanation:** same as in the Raisin notebook: mean, variance, covariance and correlation of the numerical features.

In [ ]:
# Analysis of the categorical feature Sex

# (a) Sex vs age group (two categorical variables) -> chi-square
age_group = pd.cut(df_clean["Rings"], bins=[0, 8, 10, 30], labels=["young", "middle", "old"])
table = pd.crosstab(df_clean["Sex"], age_group)
chi2, p, _, _ = chi2_contingency(table)
print(table)
print(f"Chi-square p-value = {p:.4g}")

# (b) Sex vs Rings (categorical vs numeric) -> ANOVA
groups = [g["Rings"].values for _, g in df_clean.groupby("Sex")]
f_stat, p_anova = f_oneway(*groups)
print(f"ANOVA p-value = {p_anova:.4g}")

sns.boxplot(data=df_clean, x="Sex", y="Rings")
plt.savefig("../figures/abalone_sex_rings.png", dpi=150)
plt.show()

**Explanation:** for a categorical variable use a *frequency table + chi-square* (association with another categorical variable) and *ANOVA/boxplot* (association with a numerical variable). A small p-value (< 0.05) means the relationship is statistically significant.

**TODO:** comment on the results (for example, the Infant group has clearly fewer rings).

## 9. Strongest correlated pair of features

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Abalone - correlation matrix")
plt.tight_layout()
plt.savefig("../figures/abalone_corr_heatmap.png", dpi=150)
plt.show()

pairs = corr.abs().where(~np.eye(len(corr), dtype=bool)).stack()
best_pair = pairs.idxmax()
print("Strongest pair:", best_pair, "-> r =", round(corr.loc[best_pair], 3))

**TODO:** comment on the heatmap (size and weight features are usually very highly correlated with each other, i.e. there is *multicollinearity*, which is exactly why PCA is useful).

## 10. Summary of Part 1
**TODO:** summarize the main findings and the effect of cleaning / integration / transformation.

# Part 2: PCA

## 11. Applying PCA

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
print("Number of components:", X_pca.shape[1])

**TODO:** explain the necessary preprocessing (standardization) and how the PCs are computed and selected (as in the Raisin notebook).

## 12. Explained variance

In [ ]:
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(evr))],
    "explained_variance_ratio": evr.round(4),
    "cumulative": cum.round(4),
})
print(table)

plt.figure(figsize=(7, 4))
plt.bar(range(1, len(evr) + 1), evr)
plt.plot(range(1, len(evr) + 1), cum, marker="o", color="red")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Abalone - explained variance")
plt.tight_layout()
plt.savefig("../figures/abalone_explained_variance.png", dpi=150)
plt.show()

**TODO:** compare the PCs with high and low variance. (Hint: in Abalone PC1 usually takes a very large share because all the features measure the "size" of the animal.)

## 13. Varying the number of components

In [ ]:
for k in range(1, X_scaled.shape[1] + 1):
    p = PCA(n_components=k)
    Xk = p.fit_transform(X_scaled)
    X_back = p.inverse_transform(Xk)
    error = np.mean((X_scaled - X_back) ** 2)
    print(f"k={k}: keeps {p.explained_variance_ratio_.sum():.1%} of the variance, reconstruction error = {error:.4f}")

**TODO:** comment on the trade-off between the number of dimensions and the information kept.

## 14. 2D visualization (PC1 vs PC2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Color by Sex
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_clean["Sex"].values, alpha=0.5, ax=axes[0])
axes[0].set_title("Colored by Sex")

# Color by age group (derived from Rings)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=age_group.values, alpha=0.5, ax=axes[1])
axes[1].set_title("Colored by age group")

for ax in axes:
    ax.set_xlabel(f"PC1 ({evr[0]:.1%})")
    ax.set_ylabel(f"PC2 ({evr[1]:.1%})")

plt.tight_layout()
plt.savefig("../figures/abalone_pca_2d.png", dpi=150)
plt.show()

**Explanation:** two plots side by side: one colored by `Sex`, one by age group (derived from `Rings`).

**TODO:** comment on the clusters and overlaps (usually Infants separate fairly well, while M and F overlap a lot).

In [ ]:
loadings = pd.DataFrame(pca.components_[:2].T, index=feature_cols, columns=["PC1", "PC2"])
loadings.round(3)

## 15. Original data vs PCA representation
**TODO:** discuss the advantages (dimensionality reduction, decorrelation, visualization) and the limitations (information loss, harder interpretation, only linear, sensitivity to outliers/scale).